In [1]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,seaborn as sns,warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_json('../dataset/Cleaned_Data.json')
df.head()

,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,...,Festival,City,Ordered_Hour,Ordered_Minutes,Picked_Hour,Picked_Minutes,Order_Day,Order_Month,Order_Day_of_Week,Time_taken
0,37,4.9,22.745049,75.892471,22.765049,75.912471,Sunny,High,2,Snack,...,0,Urban,11,33,11,45,19,3,5,24
1,34,4.5,12.913041,77.683237,13.043041,77.813237,Stormy,Jam,2,Snack,...,0,Metropolitian,19,45,19,51,25,3,4,33
2,23,4.4,12.914264,77.678400,12.924264,77.688400,Sandstorms,Low,0,Drinks,...,0,Urban,8,32,8,48,19,3,5,26
3,38,4.7,11.003669,76.976494,11.053669,77.026494,Sunny,Medium,0,Buffet,...,0,Metropolitian,18,3,18,12,5,4,1,21
4,32,4.6,12.972793,80.249982,13.012793,80.289982,Cloudy,High,1,Snack,...,0,Metropolitian,13,34,13,45,26,3,5,30


In [3]:
from haversine import haversine, Unit

restaurant = (28.6139, 77.2090)
delivery = (28.5355, 77.3910)

distance = haversine(restaurant, delivery, unit=Unit.KILOMETERS)

print(distance)

19.795413391165948


In [4]:
df.shape[0]

41953

In [5]:
df['Delivery_location_latitude'][1]

np.float64(13.043041)

In [6]:
i=0
while (i<df.shape[0]):
    Delivery_location_latitude=df['Delivery_location_latitude'][i]
    Delivery_location_longitude=df['Delivery_location_longitude'][i]
    Restaurant_latitude=df['Restaurant_latitude'][i]
    Restaurant_longitude=df['Restaurant_longitude'][i]

    restaurant=(Restaurant_latitude,Restaurant_longitude)
    delivery=(Delivery_location_latitude,Delivery_location_longitude)

    distance = haversine(restaurant, delivery, unit=Unit.KILOMETERS)

    df.loc[i,'Distance']=distance
    i+=1 

In [7]:
df.head()

,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,...,City,Ordered_Hour,Ordered_Minutes,Picked_Hour,Picked_Minutes,Order_Day,Order_Month,Order_Day_of_Week,Time_taken,Distance
0,37,4.9,22.745049,75.892471,22.765049,75.912471,Sunny,High,2,Snack,...,Urban,11,33,11,45,19,3,5,24,3.025153
1,34,4.5,12.913041,77.683237,13.043041,77.813237,Stormy,Jam,2,Snack,...,Metropolitian,19,45,19,51,25,3,4,33,20.183558
2,23,4.4,12.914264,77.678400,12.924264,77.688400,Sandstorms,Low,0,Drinks,...,Urban,8,32,8,48,19,3,5,26,1.552760
3,38,4.7,11.003669,76.976494,11.053669,77.026494,Sunny,Medium,0,Buffet,...,Metropolitian,18,3,18,12,5,4,1,21,7.790412
4,32,4.6,12.972793,80.249982,13.012793,80.289982,Cloudy,High,1,Snack,...,Metropolitian,13,34,13,45,26,3,5,30,6.210147


In [8]:
X=df.drop("Time_taken",axis=1)
y=df['Time_taken']

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.30,random_state=42)

In [9]:
scale_features=[i for i in X.columns if df[i].dtype!='O']
ohe_features=['Weatherconditions','Type_of_order','Type_of_vehicle']
ord_features=['Road_traffic_density','City']
traffic_order=['Low', 'Medium', 'High', 'Jam']
city_order =['Semi-Urban', 'Urban', 'Metropolitian']

In [10]:
from sklearn.preprocessing import OrdinalEncoder,StandardScaler,OneHotEncoder
scaler=StandardScaler()
ord=OrdinalEncoder(categories=[traffic_order,city_order],handle_unknown='use_encoded_value',unknown_value=-1)
ohe=OneHotEncoder(handle_unknown='ignore')

from sklearn.compose import ColumnTransformer
ct=ColumnTransformer(
    [
        ("Standard Scaler",scaler,scale_features),
        ("Ordinal Encoding",ord,ord_features),
        ("One Hot Encoding",ohe,ohe_features)
    ],
    remainder='passthrough'
)
X_train=ct.fit_transform(X_train)
X_test=ct.transform(X_test)

In [11]:
pd.DataFrame(X_test)

,0,1,2,3,4,5,6,7,8,9,...,23,24,25,26,27,28,29,30,31,32
0,0.422019,-1.632225,1.193723,1.403837,1.183523,1.388419,-1.219252,-1.321243,-0.142318,-1.567795,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-0.456749,0.823214,1.449997,-0.322374,1.452520,-0.317638,-0.026084,-1.321243,-0.142318,1.161607,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,-0.632502,-0.097575,-1.209075,-0.086375,-1.205650,-0.081659,1.167084,-1.321243,-0.142318,0.111837,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.246266,1.130144,-0.270341,0.416610,-0.270894,0.415567,-0.026084,-1.321243,-0.142318,1.161607,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,1.652294,-0.711435,0.173443,-0.452284,0.178229,-0.444678,1.167084,0.440674,-0.142318,0.741699,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12581,0.597773,1.130144,1.461307,-0.325655,1.451016,-0.340925,-0.026084,-1.321243,-0.142318,-1.567795,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
12582,-0.808256,0.516284,0.787854,0.143343,0.783282,0.136607,1.167084,0.440674,-0.142318,-0.308071,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
12583,0.246266,0.823214,-1.207239,-0.088651,-1.207474,-0.089651,1.167084,0.440674,-0.142318,0.321791,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
12584,-0.632502,0.209354,0.035375,-1.149009,0.038378,-1.144201,-1.219252,0.440674,-0.142318,1.161607,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [12]:
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

models={
    "Linear Regression":LinearRegression(),
    "Ridge":Ridge(),
    "Lasso":Lasso(),
    "SVM":SVR(),
    "KNeighborsRegressor":KNeighborsRegressor(),
    "DecisionTreeRegressor":DecisionTreeRegressor(),
    "RandomForestRegressor":RandomForestRegressor()
}

In [13]:
# from sklearn.metrics import r2_score

# for i in range(len(list(models))):
#     model_name=list(models)[i]
#     model=list(models.values())[i]
#     model.fit(X_train,y_train)
#     y_pred=model.predict(X_test)

#     print(f"{model_name}\nR^2 Score: {r2_score(y_test,y_pred)}\n")


#     Linear Regression
# R^2 Score: 0.567166674092904

# Ridge
# R^2 Score: 0.5671716146363137

# Lasso
# R^2 Score: 0.45527211702764714

# SVM
# R^2 Score: 0.6820029119283363

# KNeighborsRegressor
# R^2 Score: 0.5685970418447204

# DecisionTreeRegressor
# R^2 Score: 0.6463025491549603

# RandomForestRegressor
# R^2 Score: 0.8103044189543422


# Dropping Coordinates

In [14]:
df.drop(columns=['Restaurant_latitude','Restaurant_longitude','Delivery_location_latitude','Delivery_location_longitude'],axis=1,inplace=True)

In [15]:
X=df.drop("Time_taken",axis=1)
y=df['Time_taken']

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.30,random_state=42)

In [16]:
scale_features=[i for i in X.columns if df[i].dtype!='O']
ohe_features=['Weatherconditions','Type_of_order','Type_of_vehicle']
ord_features=['Road_traffic_density','City']
traffic_order=['Low', 'Medium', 'High', 'Jam']
city_order =['Semi-Urban', 'Urban', 'Metropolitian']

In [17]:
pd.DataFrame(X_train).columns

Index(['Delivery_person_Age', 'Delivery_person_Ratings', 'Weatherconditions',
       'Road_traffic_density', 'Vehicle_condition', 'Type_of_order',
       'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City',
       'Ordered_Hour', 'Ordered_Minutes', 'Picked_Hour', 'Picked_Minutes',
       'Order_Day', 'Order_Month', 'Order_Day_of_Week', 'Distance'],
      dtype='object')

In [18]:
from sklearn.preprocessing import OrdinalEncoder,StandardScaler,OneHotEncoder
scaler=StandardScaler()
ord=OrdinalEncoder(categories=[traffic_order,city_order],handle_unknown='use_encoded_value',unknown_value=-1)
ohe=OneHotEncoder(handle_unknown='ignore')

from sklearn.compose import ColumnTransformer
ct=ColumnTransformer(
    [
        ("Standard Scaler",scaler,scale_features),
        ("Ordinal Encoding",ord,ord_features),
        ("One Hot Encoding",ohe,ohe_features)
    ],
    remainder='passthrough'
)
X_train=ct.fit_transform(X_train)
X_test=ct.transform(X_test)

In [19]:
pd.DataFrame(X_test)

,0,1,2,3,4,5,6,7,8,9,...,19,20,21,22,23,24,25,26,27,28
0,0.422019,-1.632225,-1.219252,-1.321243,-0.142318,-1.567795,-0.358469,-1.339452,0.533756,0.365443,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-0.456749,0.823214,-0.026084,-1.321243,-0.142318,1.161607,-0.598500,1.099482,-0.203583,-0.886311,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,-0.632502,-0.097575,1.167084,-1.321243,-0.142318,0.111837,0.661662,0.161430,1.100940,-1.341495,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.246266,1.130144,-0.026084,-1.321243,-0.142318,1.161607,-0.718515,1.099482,-0.146865,-0.772515,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,1.652294,-0.711435,1.167084,0.440674,-0.142318,0.741699,-1.018553,0.724261,-0.600612,0.024056,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12581,0.597773,1.130144,-0.026084,-1.321243,-0.142318,-1.567795,1.621785,-1.151842,-1.394670,0.365443,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
12582,-0.808256,0.516284,1.167084,0.440674,-0.142318,-0.308071,-0.358469,-0.213790,0.080009,0.365443,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
12583,0.246266,0.823214,1.167084,0.440674,-0.142318,0.321791,0.181601,0.349041,0.703911,1.844790,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
12584,-0.632502,0.209354,-1.219252,0.440674,-0.142318,1.161607,0.781678,1.099482,1.271095,-0.431128,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [20]:
from sklearn.metrics import r2_score
model=RandomForestRegressor()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
print(r2_score(y_test,y_pred))


0.8122056583971915


In [21]:
from sklearn.metrics import r2_score
forest_params = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}
from sklearn.model_selection import RandomizedSearchCV
rand=RandomizedSearchCV(estimator=RandomForestRegressor(),param_distributions=forest_params,verbose=3,n_jobs=-1,cv=5)
rand.fit(X_train,y_train)
y_pred=rand.predict(X_test)
print(f"R^2 Score: {r2_score(y_test,y_pred)}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
R^2 Score: 0.8170450530544622


In [22]:
import pickle
pickle.dump(rand.best_estimator_,open('../models/model.pkl','wb'))
pickle.dump(ct,open('../models/preprocessor.pkl','wb'))